# DAiSEE AU Feature Extraction — Google Colab A100

**Sebelum mulai:**
1. Runtime → Change runtime type → **A100 GPU**
2. Dataset DAiSEE ada di Google Drive kamu di:
   `My Drive/DAiSEE/DAiSEE_extracted/DAiSEE/`
3. Jalankan sel dari atas ke bawah

**Resume:** Kalau sesi Colab putus, cukup jalankan ulang dari Sel 4 — tidak dari awal.

In [1]:
# Sel 1: Verifikasi GPU
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                    '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', r.stdout.strip())
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

GPU: NVIDIA GeForce RTX 4060 Laptop GPU, 8188 MiB


CUDA: True
Device: NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
# Sel 2: Install dependencies (jalankan sekali per sesi)
!pip install -q feat deepface tf-keras opencv-python-headless \
               scikit-learn xgboost pandas numpy scipy tqdm joblib

# Fix py-feat recursion bug dengan pandas 2.x
import sys, subprocess
feat_path = subprocess.check_output(
    [sys.executable, '-c', 'import feat.data; print(feat.data.__file__)']
).decode().strip()
print('Patching:', feat_path)

with open(feat_path, 'r') as f:
    src = f.read()

OLD = ('        # Set _metadata attributes on series: Kludgy solution\n'
       '        for k in self:\n'
       '            self[k].sampling_freq = self.sampling_freq\n'
       '            self[k].sessions = self.sessions')
NEW = ('        # Set _metadata attributes on series: Kludgy solution\n'
       "        if not getattr(self, '_in_fex_init', False):\n"
       '            try:\n'
       "                object.__setattr__(self, '_in_fex_init', True)\n"
       '                for k in self:\n'
       '                    self[k].sampling_freq = self.sampling_freq\n'
       '                    self[k].sessions = self.sessions\n'
       '            finally:\n'
       "                object.__setattr__(self, '_in_fex_init', False)")

if OLD in src:
    with open(feat_path, 'w') as f:
        f.write(src.replace(OLD, NEW))
    print('Patch applied OK')
elif '_in_fex_init' in src:
    print('Already patched')
else:
    print('WARNING: patch target not found — check py-feat version')

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/home/khai/Daisee/venv/lib/python3.12/site-packages/feat/data.py", line 42, in <module>
    from feat.utils.io import read_feat, read_openface
  File "/home/khai/Daisee/venv/lib/python3.12/site-packages/feat/utils/io.py", line 9, in <module>
    from feat.utils import (
ImportError: cannot import name 'FEAT_EMOTION_COLUMNS' from 'feat.utils' (/home/khai/Daisee/venv/lib/python3.12/site-packages/feat/utils/__init__.py)


CalledProcessError: Command '['/home/khai/Daisee/venv/bin/python', '-c', 'import feat.data; print(feat.data.__file__)']' returned non-zero exit status 1.

In [ ]:
# Sel 3: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Sesuaikan kalau struktur folder Drive kamu berbeda
DRIVE_ROOT  = Path('/content/drive/MyDrive/DAiSEE')
DAISEE_ROOT = DRIVE_ROOT / 'DAiSEE_extracted' / 'DAiSEE'
DATASET_DIR = DAISEE_ROOT / 'DataSet'
LABELS_DIR  = DAISEE_ROOT / 'Labels'
OUTPUT_DIR  = DRIVE_ROOT / 'au_features'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for p in [DATASET_DIR, LABELS_DIR]:
    print('OK  ' if p.exists() else 'MISSING  ', p)
print('Output:', OUTPUT_DIR)

In [ ]:
# Sel 4: Config & Load Detector
# Sesuaikan BATCH_SIZE:
#   A100 80GB → 32 | A100 40GB → 16 | V100 → 8 | T4 → 4
BATCH_SIZE  = 16
DEVICE      = 'cuda'
FPS_TARGET  = 1
ROLLING_WIN = 3

AU_COLS = [
    'AU01','AU02','AU04','AU05','AU06','AU07','AU09','AU10',
    'AU11','AU12','AU14','AU15','AU17','AU20','AU23','AU24',
    'AU25','AU26','AU28','AU43',
]
SPLIT_MAP = {
    'train': ('Train',      'TrainLabels.csv'),
    'test':  ('Test',       'TestLabels.csv'),
    'val':   ('Validation', 'ValidationLabels.csv'),
}
BROKEN_VIDEOS = {'2100552061.avi'}

from feat import Detector
print('Loading py-feat detector...')
det = Detector(
    au_model       = 'xgb',
    face_model     = 'retinaface',
    landmark_model = 'mobilefacenet',
    emotion_model  = 'resmasknet',
    facepose_model = 'img2pose',
    device         = DEVICE,
)
print('Detector ready.')

In [ ]:
# Sel 5: Helper functions
import cv2, tempfile, time
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

def extract_one(video_path):
    cap  = cv2.VideoCapture(str(video_path))
    fps  = cap.get(cv2.CAP_PROP_FPS) or 30.0
    skip = max(1, int(round(fps / FPS_TARGET)))
    frame_paths, timestamps = [], []
    tmpdir = tempfile.mkdtemp()
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        if idx % skip == 0:
            p = os.path.join(tmpdir, f'f{idx:05d}.png')
            cv2.imwrite(p, frame)
            frame_paths.append(p)
            timestamps.append(idx / fps)
        idx += 1
    cap.release()
    if not frame_paths:
        _cleanup(tmpdir, frame_paths); return None
    try:
        result = det.detect_image(frame_paths, output_size=(640, 480),
                                   batch_size=BATCH_SIZE)
    except Exception as e:
        print(f'  [warn] {Path(video_path).name}: {e}')
        _cleanup(tmpdir, frame_paths); return None
    _cleanup(tmpdir, frame_paths)
    au_present = [c for c in AU_COLS if c in result.columns]
    if not au_present: return None
    # Satu baris per frame — ambil face dengan FaceScore tertinggi
    result = result.reset_index(drop=True)
    if 'input' in result.columns:
        sc = 'FaceScore' if 'FaceScore' in result.columns else None
        if sc:
            result = (result.sort_values(sc, ascending=False)
                            .groupby('input', sort=False).first()
                            .reset_index(drop=True))
        else:
            result = result.groupby('input', sort=False).first().reset_index(drop=True)
    n  = min(len(result), len(timestamps))
    df = result[au_present].iloc[:n].copy().reset_index(drop=True)
    df.insert(0, 'timestamp', timestamps[:n])
    df[au_present] = df[au_present].ffill().bfill()
    if df[au_present].isna().all(axis=None): return None
    for col in AU_COLS:
        if col not in df.columns: df[col] = 0.0
    df[AU_COLS] = df[AU_COLS].rolling(window=ROLLING_WIN, min_periods=1).mean()
    return df[['timestamp'] + AU_COLS]

def _cleanup(tmpdir, fps):
    for p in fps:
        try: os.unlink(p)
        except: pass
    try: os.rmdir(tmpdir)
    except: pass

def load_done(path):
    if not path.exists(): return set()
    try: return set(pd.read_csv(path, usecols=['video_id'])['video_id'].unique())
    except: path.unlink(); return set()

print('Helpers loaded.')

In [ ]:
# Sel 6: Ekstraksi AU features
# Kalau sesi putus, jalankan ulang sel ini saja — otomatis resume

SPLITS_TO_RUN = ['train', 'test', 'val']

for split in SPLITS_TO_RUN:
    split_folder, label_file = SPLIT_MAP[split]
    split_dir   = DATASET_DIR / split_folder
    output_path = OUTPUT_DIR / f'{split}_au_features.csv'

    labels = pd.read_csv(LABELS_DIR / label_file)
    labels.columns = labels.columns.str.strip()
    clip_ids  = labels['ClipID'].tolist()
    done      = load_done(output_path)
    remaining = [c for c in clip_ids if c not in done and c not in BROKEN_VIDEOS]

    print(f'\n[{split}] {len(remaining)} to process (done={len(done)}, total={len(clip_ids)})')

    file_mode    = 'a' if output_path.exists() else 'w'
    write_header = not output_path.exists()
    ok = no_face = missing = 0
    t0 = time.perf_counter()

    with open(output_path, file_mode, buffering=1) as fout:
        pbar = tqdm(remaining, desc=split, unit='vid')
        for clip_id in pbar:
            matches = list(split_dir.rglob(clip_id))
            if not matches:
                missing += 1; continue
            df = extract_one(matches[0])
            if df is None:
                no_face += 1
            else:
                df.insert(0, 'video_id', clip_id)
                df.to_csv(fout, index=False, header=write_header)
                write_header = False
                ok += 1
            elapsed = time.perf_counter() - t0
            rate    = (ok + no_face + missing) / max(elapsed, 1)
            eta_h   = (len(remaining) - ok - no_face - missing) / max(rate, 1e-6) / 3600
            pbar.set_postfix(ok=ok, no_face=no_face, miss=missing, eta=f'{eta_h:.1f}h')

    print(f'[{split}] selesai — ok={ok}, no_face={no_face}, missing={missing}')

print('\nSemua split selesai!')

In [ ]:
# Sel 7: Aggregate per video
for split in SPLITS_TO_RUN:
    src = OUTPUT_DIR / f'{split}_au_features.csv'
    dst = OUTPUT_DIR / f'{split}_au_features_agg.csv'
    if not src.exists():
        print(f'skip {src.name}'); continue
    df  = pd.read_csv(src)
    agg = df.groupby('video_id')[AU_COLS].mean().reset_index()
    agg.to_csv(dst, index=False)
    print(f'[agg] {split}: {len(agg)} videos → {dst.name}')

print('\nFile tersimpan di Google Drive:', OUTPUT_DIR)